# Greedy Decoding

In [6]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Load GPT-2
tok = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

out = model.generate(
    **tok("Rose loves ", return_tensors="pt").to(model.device),
    max_new_tokens=2,
    do_sample=False
)
print(tok.decode(out[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Rose loves iced tea


# Top -K 

In [19]:
import re
import torch

# Build a one-time allowlist of token IDs that look like words:
# - Leading space followed by letters, possibly with hyphen/apostrophe letters after.
wordlike_ids = []
for tok_str, tok_id in tok.get_vocab().items():
    if re.match(r"^([Ġ ]|^)[A-Za-z][A-Za-z'\-]*$", tok_str):
        wordlike_ids.append(tok_id)
wordlike_ids = set(wordlike_ids)

def topk_debug_readable_filtered(prefix, k=5, temperature=1.0):
    model.eval()
    device = next(model.parameters()).device
    enc = tok(prefix, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = model(**enc).logits[:, -1, :]

    # Mask non-wordlike tokens
    mask = torch.full_like(logits, float("-inf"))
    mask[:, list(wordlike_ids)] = logits[:, list(wordlike_ids)]
    logits = mask

    logits = logits / temperature
    probs  = torch.softmax(logits, dim=-1)
    topk   = torch.topk(probs, k=k)
    ids    = topk.indices[0].tolist()
    ps     = topk.values[0].tolist()

    base_ids = enc["input_ids"][0].tolist()

    print(f"\nPrefix: {repr(prefix)} (filtered)")
    print("Top-k candidates (token_piece → human continuation):")
    for rank, (tid, p) in enumerate(zip(ids, ps), 1):
        piece = tok.convert_ids_to_tokens([tid])[0]
        cont  = tok.decode(base_ids + [tid], skip_special_tokens=True)[len(prefix):]
        print(f"{rank}. {piece!r:<12} prob={p:.4f}  ⇒ {cont!r}")

    # sample from filtered top-k
    choice_idx = torch.multinomial(topk.values, 1).item()
    chosen_id  = ids[choice_idx]
    out_ids    = torch.tensor(base_ids + [chosen_id]).to(device)
    chosen_text = tok.decode(out_ids, skip_special_tokens=True)

    print(f"\nSampled choice → {choice_idx+1}")
    print("Continuation:", chosen_text)
topk_debug_readable_filtered("Rose loves", k=5, temperature=2.0)



Prefix: 'Rose loves' (filtered)
Top-k candidates (token_piece → human continuation):
1. 'Ġto'        prob=0.0067  ⇒ ' to'
2. 'Ġthe'       prob=0.0044  ⇒ ' the'
3. 'Ġit'        prob=0.0033  ⇒ ' it'
4. 'Ġher'       prob=0.0032  ⇒ ' her'
5. 'Ġhis'       prob=0.0030  ⇒ ' his'

Sampled choice → 2
Continuation: Rose loves the


In [21]:
import torch
import re

# Build once (same regex you used; relax if it's too strict)
wordlike_ids = []
for tok_str, tok_id in tok.get_vocab().items():
    if re.match(r"^([Ġ ]|^)[A-Za-z][A-Za-z'\-]*$", tok_str):
        wordlike_ids.append(tok_id)
wordlike_ids = set(wordlike_ids)

def topp_debug_readable(prefix, p=0.9, temperature=1.0, n_draws=1, replacement=None):
    model.eval()
    device = next(model.parameters()).device
    enc = tok(prefix, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = model(**enc).logits[:, -1, :]

    # Filter non-wordlike tokens
    mask = torch.full_like(logits, float("-inf"))
    idxs = list(wordlike_ids)
    mask[:, idxs] = logits[:, idxs]
    logits = mask

    # Temperature + probs
    logits = logits / temperature
    probs = torch.softmax(logits, dim=-1).squeeze()

    # If everything got masked (all -inf -> NaNs), fall back to unfiltered
    if not torch.isfinite(probs).any():
        probs = torch.softmax((model(**enc).logits[:, -1, :] / temperature).squeeze(), dim=-1)

    # Sort and take nucleus
    sorted_probs, sorted_ids = torch.sort(probs, descending=True)
    cum = torch.cumsum(sorted_probs, dim=-1)
    keep_mask = cum <= p
    # Ensure at least one token is kept
    if not keep_mask.any():
        keep_mask[0] = True
    keep_ids = sorted_ids[keep_mask]
    kept_probs = probs[keep_ids]
    kept_probs = kept_probs / kept_probs.sum()

    base_ids = enc["input_ids"][0].tolist()
    print(f"\nPrefix: {repr(prefix)}   top-p={p}, temperature={temperature}")
    print("Nucleus candidates (piece → continuation):")
    for rank, (tid, pr) in enumerate(zip(keep_ids.tolist(), kept_probs.tolist()), 1):
        piece = tok.convert_ids_to_tokens([tid])[0]
        cont  = tok.decode(base_ids + [tid], skip_special_tokens=True)[len(prefix):]
        print(f"{rank:>2}. {piece!r:<12} prob={pr:.4f}  ⇒ {cont!r}")

    # Decide replacement
    if replacement is None:
        # If requesting more than we have, switch to replacement=True automatically
        replacement = n_draws > keep_ids.numel()

    # Cap n_draws when sampling without replacement
    actual_draws = n_draws if replacement else min(n_draws, keep_ids.numel())

    # Sample
    choice_idx = torch.multinomial(kept_probs, actual_draws, replacement=replacement)
    if actual_draws == 1:
        choice_idx = choice_idx.unsqueeze(0)
    print("\nSampled indices (1-based in the printed list):", [i.item()+1 for i in choice_idx])

    # Show continuations for each draw
    for i, idx in enumerate(choice_idx.tolist(), 1):
        chosen_id = keep_ids[idx].item()
        cont_text = tok.decode(base_ids + [chosen_id], skip_special_tokens=True)
        print(f"Pick #{i}: {tok.convert_ids_to_tokens([chosen_id])[0]!r} → {cont_text}")

    return keep_ids, kept_probs, choice_idx
topp_debug_readable("Rose loves ", p=0.9, temperature=2.0, n_draws=5)



Prefix: 'Rose loves '   top-p=0.9, temperature=2.0
Nucleus candidates (piece → continuation):
 1. 'iced'       prob=0.0258  ⇒ 'iced'
 2. 'vern'       prob=0.0089  ⇒ 'vern'
 3. 'ich'        prob=0.0083  ⇒ 'ich'
 4. 'ik'         prob=0.0067  ⇒ 'ik'
 5. 'icky'       prob=0.0064  ⇒ 'icky'
 6. 'iz'         prob=0.0055  ⇒ 'iz'
 7. 'urch'       prob=0.0054  ⇒ 'urch'
 8. 'urs'        prob=0.0051  ⇒ 'urs'
 9. 'urn'        prob=0.0047  ⇒ 'urn'
10. 'ive'        prob=0.0043  ⇒ 'ive'
11. 'xt'         prob=0.0042  ⇒ 'xt'
12. 'ike'        prob=0.0040  ⇒ 'ike'
13. 'ery'        prob=0.0039  ⇒ 'ery'
14. 'ers'        prob=0.0039  ⇒ 'ers'
15. 'iph'        prob=0.0036  ⇒ 'iph'
16. 'ix'         prob=0.0036  ⇒ 'ix'
17. 'ices'       prob=0.0035  ⇒ 'ices'
18. 'irl'        prob=0.0034  ⇒ 'irl'
19. 'ick'        prob=0.0031  ⇒ 'ick'
20. 'ian'        prob=0.0030  ⇒ 'ian'
21. 'ile'        prob=0.0022  ⇒ 'ile'
22. 'irc'        prob=0.0021  ⇒ 'irc'
23. 'ikan'       prob=0.0021  ⇒ 'ikan'
24. 'ili'        prob=0.0019 

(tensor([ 3711,   933,   488,  ..., 49579, 42097, 28785]),
 tensor([2.5846e-02, 8.9382e-03, 8.3380e-03,  ..., 9.4915e-06, 9.4913e-06,
         9.4900e-06]),
 tensor([ 135, 4417, 3283, 4297, 9164]))